## Modelos Baseline SARIMA y SARIMAX — Sistema de Predicción Energética en Barcelona

> **Contexto:** A partir de `dataset_features` (notebooks 03 y 04), se construyen los modelos
> **baseline** del TFM: **SARIMA** (univariante) y **SARIMAX** (con variables exógenas). Son la
> referencia contra la que se compararán XGBoost/LightGBM y LSTM/GRU en fases posteriores.

### Directiva de modelado (tutoría José, obligatoria)

- Baseline = **SARIMA y SARIMAX** únicamente (Prophet descartado).
- **Test = 2025 más reciente** (oct–nov, datos terminan 2025-11-30); **nunca se toca**.
- Métrica **primaria = R²** (maximizar); reportar también MAE, RMSE, MAPE.
- Grid search: **NO usar `best_model` directo** → guardar todo en CSV, calcular la diferencia
  relativa R²_train vs R²_val, graficar overfitting y elegir **mejor R²_val con rel_diff ≤ 0.10**.
- Documentar **ADF + KPSS**.

### Estrategia de validación (train / validación / test)

| Conjunto | Fechas | Uso |
|---|---|---|
| **Train** | 2019-01 → 2024-12 | Ajuste + R²_train (in-sample) |
| **Validación** | 2025-01 → 2025-09 | Grid search: R²_val + control de overfitting |
| **Test** | 2025-10 → 2025-11 | Backtesting final out-of-sample (intacto) |

- **Walk-forward backtesting** (igual que skforecast en las fuentes py29/py51), horizonte **72h = 12 bloques**.
- **Grid search** sobre los **4 CPs representativos** (08038, 08005, 08032, 08002); el backtest final
  se aplica a los **42 CPs** con los órdenes seleccionados.
- Órdenes seed del EDA (ACF/PACF): `d=0, D=1, s=28; p≈3-4, q≈1, P=Q=1`.

> ⚠️ **Caveat (presente, no tratado aún):** el target del CP **08037** (jul–nov 2025) es imputado
> (sintético) y cae dentro de validación y test. Sus métricas no reflejan demanda real; se decidirá
> más adelante si se trata aparte o se excluye.

## Librerías

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pymongo import MongoClient
import warnings, time

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#D3D3D3'
plt.rcParams['grid.linewidth'] = 0.4
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Paleta de color
C1 = '#264653'; C2 = '#2A9D8F'; C3 = '#E9C46A'
C4 = '#F4A261'; C5 = '#E76F51'; C6 = '#A8DADC'
TITULO = '#1B3A5C'; SUBTITULO = '#C0392B'

start_time = time.time()

In [ ]:
client = MongoClient('mongodb://mongo:27017/')
db     = client['tfm_energy']

docs = list(db['dataset_features'].find({}, {'_id': 0}))
df   = pl.DataFrame(docs, infer_schema_length=None)

print(f"Shape: {df.shape}")
print(f"Desde: {df['datetime'].min()}  Hasta: {df['datetime'].max()}")
print(f"Codigos postales: {df['cod_postal'].n_unique()}")

---
# <font color='#1B3A5C'>  **1. Configuración del Experimento** </font>

> Se fijan las fechas de corte, el horizonte, los CPs y las funciones que extraen cada serie.
> Estos parámetros son **idénticos para todos los modelos** (SARIMA, SARIMAX y los de fases
> posteriores), lo que garantiza una comparación directa y justa.

In [ ]:
# Cortes temporales (indice de bloques de 6h)
FIN_TRAIN = '2024-12-31 18:00:00'   # fin de train
FIN_VAL   = '2025-09-30 18:00:00'   # fin de validacion (2025-01 -> 2025-09)
# test: 2025-10-01 -> 2025-11-30 (lo que queda tras FIN_VAL), INTACTO

# Horizonte de prediccion
STEPS = 12          # 72h (principal).  STEPS = 4 -> 24h (alternativo)
S = 28              # periodo estacional semanal (4 bloques/dia * 7)

# Ordenes seed del EDA (ACF/PACF)
SEED_ORDER    = (3, 0, 1)
SEED_SORDER   = (1, 1, 1, S)

# Historico COMPLETO de entrenamiento (identico al notebook 06 -> comparacion justa).
INI = '2019-01-01'                    # train arranca en 2019 (sin recortar)
FIN_VAL_GRID = FIN_VAL                 # el grid valida ene-sep 2025 (completo)

# CPs - SOLO LIMPIOS (auditoria de calidad de datos, notebook 02).
# OpenData reasigna consumo entre CPs en 2024-2025; 12 CPs tienen el target corrupto
# en val/test y se excluyen del estudio (se explicara a Jose). Quedan 30 CPs limpios.
CPS_SUCIOS = ['08011','08009','08007','08013','08010','08006','08005',
              '08019','08008','08036','08026','08037']
CPS_TODOS = [cp for cp in sorted(df['cod_postal'].unique().to_list()) if cp not in CPS_SUCIOS]  # 30 limpios
CPS_REPR  = ['08038', '08032', '08002', '08027']         # 4 perfiles representativos (limpios; 08005 era sucio)
CPS_GRID  = CPS_REPR

print(f"Grid train : {INI} -> {FIN_TRAIN}  (2019-2024 completo)")
print(f"Final train: {INI} -> {FIN_VAL}")
print(f"Test       : 2025-10-01 -> {df['datetime'].max()}  (intacto)")
print(f"Horizonte  : {STEPS} bloques ({STEPS*6}h) | s = {S}")
print(f"CPs grid: {CPS_GRID} | CPs total: {len(CPS_TODOS)}")

> Órdenes seed (3,0,1)(1,1,1)28 — del EDA (03):
> - d=0: ADF estacionaria en media (sin tendencia).
> - p=3: PACF significativa hasta el lag 3-4.
> - q=1: ACF decae gradual,  MA corto.
> - s=28, D=1: ciclo semanal fuerte (STL + pico ACF lag 28 = 0.87) diferenciación estacional.
> - P=1, Q=1 son los  ecos semanales en lags 56/84, un término AR y MA estacional.

In [ ]:
# Extrae la serie mwh_total de un CP como pandas Series con frecuencia de 6h
def get_serie(cp):
    s = (df.filter(pl.col('cod_postal') == cp)
           .sort('datetime')
           .select(['datetime', 'mwh_total'])
           .to_pandas().set_index('datetime')['mwh_total'])
    return s.asfreq('6h')

# Comprobación rápida
_s = get_serie('08002')
print(f"08002 -> {len(_s)} bloques, nulos: {_s.isna().sum()}")
print(_s.head(3))

---
# <font color='#1B3A5C'>  **2. Funciones del Arnés de Evaluación** </font>

> Tres funciones reutilizables que definen la comparación: el cálculo de métricas (R² primaria),
> el ajuste de un SARIMAX y el **backtesting walk-forward**. El backtest ajusta el modelo una vez
> sobre el train y luego avanza por el periodo de evaluación actualizando el estado con las
> observaciones reales (`append`), tal como recomienda la fuente py51 para modelos estadísticos.

In [ ]:
def metricas(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    denom = np.where(y_true == 0, np.nan, y_true)
    return {
        'r2'  : r2_score(y_true, y_pred),
        'mae' : mean_absolute_error(y_true, y_pred),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mape': float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100),
    }


def ajustar_sarimax(serie, order, seasonal_order, exog=None):
    mod = SARIMAX(serie, exog=exog, order=order, seasonal_order=seasonal_order,
                  enforce_stationarity=False, enforce_invertibility=False)
    return mod.fit(disp=False)


def backtest(res, serie_eval, steps, exog_eval=None, refit=False, W=600):
    # Walk-forward ROBUSTO: re-filtra (sin re-optimizar) sobre los ultimos W puntos
    # de historia usando arrays numpy. Evita res.extend(), que falla por checks de
    # indice/freq en algunos entornos. Resultado identico al extend (verificado).
    params = res.params
    order  = res.model.order
    sorder = res.model.seasonal_order

    hist    = list(np.asarray(res.model.endog).ravel())
    hist_ex = None
    if exog_eval is not None:
        hist_ex = np.atleast_2d(np.asarray(res.model.exog)).tolist()
    ev  = np.asarray(serie_eval, dtype=float)
    evx = np.asarray(exog_eval, dtype=float) if exog_eval is not None else None

    cur = res
    y_true, y_pred = [], []
    n = len(ev); i = 0
    while i < n:
        h = min(steps, n - i)
        ex_f = evx[i:i+h] if evx is not None else None
        fc = np.asarray(cur.forecast(steps=h, exog=ex_f))[:h]
        y_pred.extend(fc); y_true.extend(ev[i:i+h])
        # crecer la historia con los reales y re-filtrar sobre los ultimos W puntos
        hist.extend(ev[i:i+h])
        if evx is not None:
            hist_ex.extend(evx[i:i+h].tolist())
        hh  = np.asarray(hist[-W:])
        hhx = np.asarray(hist_ex[-W:]) if hist_ex is not None else None
        cur = SARIMAX(hh, exog=hhx, order=order, seasonal_order=sorder,
                      enforce_stationarity=False, enforce_invertibility=False).filter(params)
        i += h
    return np.array(y_true), np.array(y_pred)

---
# <font color='#1B3A5C'>  **3. Pruebas de Estacionariedad (ADF + KPSS)** </font>

> Requisito de José: documentar ADF y KPSS. Se confirman sobre la serie de cada CP representativo
> antes de modelar (el EDA ya lo hizo a nivel global). Recordatorio:
- **ADF** H0: la serie NO es estacionaria (raíz unitaria). p < 0.05, estacionaria en media.
- **KPSS** H0: la serie SÍ es estacionaria. p < 0.05,  NO estacionaria (en varianza).

In [ ]:
filas = []
for cp in CPS_REPR:
    serie = get_serie(cp).loc[:FIN_VAL].dropna()
    adf_p  = adfuller(serie, autolag='AIC')[1]
    kpss_p = kpss(serie, regression='c', nlags='auto')[1]
    filas.append({
        'cp': cp,
        'ADF_p': round(adf_p, 4),  'ADF': 'estacionaria' if adf_p < 0.05 else 'no',
        'KPSS_p': round(kpss_p, 4), 'KPSS': 'estacionaria' if kpss_p > 0.05 else 'no',
    })
print(pd.DataFrame(filas).to_string(index=False))


> Tiene el mismo patrón que el EDA, estacionaria en media (ADF rechaza en los 4, d=0), pero con estructura que cambia en varianza/estacionalidad (KPSS rechaza, `D=1` para domar el  ciclo semanal). 

- Los dos tests se contradicen a propósito, y esa estacionariedad parcial es justo lo que justifica la configuración SARIMA.

-  08038 (industrial, Zona Franca) es el único donde KPSS no rechaza al 5%: es algo más estacionario, coherente con un consumo industrial más rutinario. No cambia la estrategia.

 todos confirman d=0; la no-estacionariedad restante es estacional D=1

---
# <font color='#1B3A5C'>  **4. Baseline: Naive Estacional** </font>

> Modelo de referencia mínimo (igual que `ForecasterEquivalentDate` en py29): predice cada bloque
> con el valor del **mismo bloque de hace 1 semana** (`shift(28)`). Si SARIMA no le gana, no aporta.

In [ ]:
filas = []
for cp in CPS_TODOS:
    serie = get_serie(cp)
    test  = serie.loc[FIN_VAL:].iloc[1:]          # oct-nov 2025
    pred  = serie.shift(S).loc[test.index]        # valor de hace 1 semana
    m = metricas(test.values, pred.values); m['cp'] = cp
    filas.append(m)

base_res = pd.DataFrame(filas)
print("Baseline naive estacional (test oct-nov 2025), resumen 30 CPs limpios")
print(base_res[['r2', 'mae', 'rmse', 'mape']].mean().round(3))
print(base_res[['r2','mae','rmse','mape']].median().round(3))   # mediana

In [ ]:
# Tabla por CP, de mejor a peor R²
pd.set_option('display.max_rows', 50)
print(base_res.sort_values('r2', ascending=False)[['cp','r2','mae','rmse','mape']].round(2).to_string(index=False))

# Distribución real (en vez de solo la media)
print("\nDistribución de R² entre los 30 CPs limpios:")
print(base_res['r2'].describe().round(3))

# 08037 (el imputado, sospechoso de arrastrar la media)
print("\n08037:", '(08037 excluido por calidad de datos, notebook 02)')

In [ ]:
b = base_res.sort_values('r2')
colores = [C5 if r < 0 else (C3 if r < 0.5 else C2) for r in b['r2']]
fig, ax = plt.subplots(figsize=(8, 11))
ax.barh(b['cp'], b['r2'], color=colores, edgecolor='white')
ax.axvline(0, color='gray', lw=0.8)
ax.set_xlabel('R²'); ax.set_title('Naive estacional  R² por código postal', fontweight='bold', color=TITULO)
plt.tight_layout(); plt.show()

---
# <font color='#1B3A5C'>  **5. SARIMA  Grid Search (4 CPs representativos)** </font>



> Se prueba una rejilla de órdenes (sembrada por el ACF/PACF del EDA) sobre los 4 perfiles.
> Para **cada** combinación se guarda **R²_train (in-sample) y R²_val (backtest)** en CSV.
> La selección sigue la regla de José: **no** se toma el mejor sin más.



In [ ]:
res_grid = pd.read_csv('grid_sarima.csv')
print(f"Grid cargado de CSV ({len(res_grid)} filas) — ganador (2,0,1)x(1,1,1,28)")

In [ ]:
# Grid (directiva 4 de Jose): varias combinaciones,
# se guarda TODO en CSV y se elige por R2_val + rel_diff (NO best_model directo).
#  ORDERS  = [(2, 0, 1), (3, 0, 1)]
#  SORDERS = [(1, 1, 1, S), (0, 1, 1, S)]
#  
#  filas = []
#  for cp in CPS_GRID:
#      serie = get_serie(cp)
#      s_tr = serie.loc[INI:FIN_TRAIN]                    # train 2019-2024 (completo)
#      s_va = serie.loc[FIN_TRAIN:FIN_VAL_GRID].iloc[1:]  # validacion ene-sep 2025
#      for order in ORDERS:
#          for so in SORDERS:
#              try:
#                  res = ajustar_sarimax(s_tr, order, so)
#                  # R2_train a 72h (in-sample, dynamic) -> mismo horizonte que R2_val
#                  ytr, ypr = [], []
#                  for st in range(S, len(s_tr) - STEPS, STEPS):
#                      p = res.predict(start=st, end=st + STEPS - 1, dynamic=True)
#                      ypr.extend(np.asarray(p)); ytr.extend(np.asarray(s_tr.iloc[st:st+STEPS]))
#                  r2_tr = r2_score(ytr, ypr)
#                  yt, yp = backtest(res, s_va, STEPS)
#                  r2_va = r2_score(yt, yp)
#                  rel = (r2_tr - r2_va) / abs(r2_tr) if r2_tr != 0 else np.nan
#                  filas.append({'cp': cp, 'order': str(order), 'seasonal': str(so),
#                                'r2_train': round(r2_tr, 4), 'r2_val': round(r2_va, 4),
#                                'rel_diff': round(rel, 4),
#                                'mae_val': round(mean_absolute_error(yt, yp), 1)})
#                  print(f"{cp} {order}x{so}: R2_tr={r2_tr:.3f} R2_val={r2_va:.3f} rel={rel:.3f}")
#              except Exception as e:
#                  filas.append({'cp': cp, 'order': str(order), 'seasonal': str(so),
#                                'error': str(e)[:60]})
#                  print(f"{cp} {order}x{so}: ERROR {str(e)[:50]}")
#  
#  res_grid = pd.DataFrame(filas)
#  res_grid.to_csv('grid_sarima.csv', index=False)
#  print(f"Guardado grid_sarima.csv ({len(res_grid)} filas)")

> **Nota — `rel_diff` justo:** el `R2_train` y el `R2_val` se miden **ambos a 72h** (walk-forward / dynamic in-sample), mismo horizonte → el `rel_diff` mide overfitting real, no un artefacto de escala. Ganador del grid: **`(2,0,1)(1,1,1,28)`** (`p=2` generaliza mejor que `p=3` en los 4 CPs).

### <font color='#C0392B'><b>5.1 Selección con control de overfitting</b></font>

> Regla de José: descartar combinaciones con `rel_diff > 0.10` (overfitting) y, entre las que
> quedan, elegir la de **mayor R²_val**. Se promedia entre los 4 CPs para fijar un orden común.

In [ ]:
UMBRAL_REL_DIFF = 0.10

ok = res_grid.dropna(subset=['r2_val']) if 'r2_val' in res_grid else res_grid
agg = (ok.groupby(['order', 'seasonal'])
         .agg(r2_val=('r2_val', 'mean'), rel_diff=('rel_diff', 'mean'),
              mae_val=('mae_val', 'mean'))
         .reset_index())

print('Todas las combinaciones (media 4 CPs), ordenadas por R2_val:')
print(agg.sort_values('r2_val', ascending=False).to_string(index=False))
print()

candidatos = agg[agg['rel_diff'] <= UMBRAL_REL_DIFF]
if candidatos.empty:
    # OJO: rel_diff sale alto porque R2_train se mide a 1 paso (in-sample) y
    # R2_val a 72h (walk-forward) -> no son comparables; no es solo overfitting.
    # Fallback: elegir por mejor R2_val y reportar el rel_diff con la decision.
    print(f'Ninguna combinacion baja de rel_diff <= {UMBRAL_REL_DIFF} '
          f'(train=1 paso vs val=72h infla el rel). Se elige por mejor R2_val.')
    candidatos = agg
else:
    print(f'Candidatos sin overfitting (rel_diff <= {UMBRAL_REL_DIFF}):')
    print(candidatos.sort_values('r2_val', ascending=False).to_string(index=False))

best = candidatos.sort_values('r2_val', ascending=False).iloc[0]
print()
print(f"Seleccionado -> order={best['order']} seasonal={best['seasonal']} "
      f"(R2_val={best['r2_val']:.3f}, rel_diff={best['rel_diff']:.3f})")

In [ ]:
# Gráfico overfitting: R2_val vs rel_diff
fig, ax = plt.subplots(figsize=(10, 6))
colores = [C5 if r > UMBRAL_REL_DIFF else C2 for r in agg['rel_diff']]
ax.scatter(agg['rel_diff'], agg['r2_val'], c=colores, s=90, edgecolors='white', zorder=3)
for _, r in agg.iterrows():
    ax.annotate(f"{r['order']}\n{r['seasonal']}", (r['rel_diff'], r['r2_val']),
                fontsize=6, alpha=0.7, ha='center')
ax.axvline(UMBRAL_REL_DIFF, color=C5, linestyle='--', linewidth=1)
ax.text(UMBRAL_REL_DIFF + 0.002, ax.get_ylim()[0], 'umbral 0.10', color=C5, fontsize=8)
ax.set_xlabel('Diferencia relativa R²_train vs R²_val (overfitting →)')
ax.set_ylabel('R²_val (media 4 CPs)')
ax.set_title('Selección de órdenes SARIMA — R²_val vs overfitting', fontweight='bold', color=TITULO)
plt.tight_layout(); plt.show()

> Verde = combinaciones aceptables (`rel_diff ≤ 0.10`); rojo = overfitting. Se elige el punto
> verde más alto. **Fija el orden seleccionado en la celda siguiente** antes del backtest de los 42 CPs.

---
# <font color='#1B3A5C'>  **6. SARIMA — Backtesting Final (30 CPs limpios)** </font>

> Con el orden seleccionado `(2,0,1)(1,1,1,28)` se ajusta un SARIMA por CP sobre
> train+validación y se hace el backtesting sobre el test (oct–nov 2025), intacto.
> Bucle secuencial (un CP tras otro); backtest robusto por ventanas (memoria ligera).

In [ ]:
# Backtesting SARIMA por CP (secuencial). Backtest robusto walk-forward.
ORDER_SEL  = (2, 0, 1)           # ganador del grid (seccion 5.1)
SORDER_SEL = (1, 1, 1, S)        # estacional del seed

filas = []
for cp in CPS_TODOS:
    serie = get_serie(cp)
    s_trv = serie.loc[INI:FIN_VAL]           # train + validacion
    s_te  = serie.loc[FIN_VAL:].iloc[1:]     # test oct-nov 2025
    try:
        res = ajustar_sarimax(s_trv, ORDER_SEL, SORDER_SEL)
        yt, yp = backtest(res, s_te, STEPS)
        m = metricas(yt, yp); m['cp'] = cp
        filas.append(m)
        print(f"{cp}: R2={m['r2']:.3f} MAE={m['mae']:.0f} MAPE={m['mape']:.1f}%")
    except Exception as e:
        print(f"{cp}: ERROR {str(e)[:50]}")

sarima_res = pd.DataFrame(filas)
print()
print('SARIMA - media 30 CPs limpios:')
print(sarima_res[['r2', 'mae', 'rmse', 'mape']].mean().round(3))

---
# <font color='#1B3A5C'>  **7. SARIMAX — Con Variables Exógenas** </font>

> SARIMAX = SARIMA + regresores externos conocidos en el momento de predecir:
- **Clima:** `temp_mean`, `HDD`, `CDD`, `humedad_mean` (supuesto: pronóstico perfecto).
- **Calendario:** `es_festivo`, `is_covid`.
- **Ciclo diario:** términos de Fourier de la hora (`sin_dia`, `cos_dia`) — capturan la
  estacionalidad diaria (s=4) que el SARIMA semanal (s=28) no modela.

> Los **lags** (lag_1…lag_28, rolling) NO entran: la autorregresión la hace SARIMA internamente.

In [ ]:
# Exogenas para SARIMAX: clima + calendario + Fourier diario
EXOG_COLS = ['HDD', 'CDD', 'humedad_mean', 'es_festivo', 'is_covid']  # temp_mean fuera: colineal con HDD/CDD

def get_exog(cp):
    e = (df.filter(pl.col('cod_postal') == cp)
           .sort('datetime')
           .select(['datetime', 'hora'] + EXOG_COLS)
           .to_pandas().set_index('datetime').asfreq('6h'))
    e = e.ffill().bfill()
    ang = 2 * np.pi * e['hora'] / 24
    e['sin_dia'] = np.sin(ang); e['cos_dia'] = np.cos(ang)
    return e.drop(columns=['hora'])

_e = get_exog('08002')
print(f"exog 08002: {_e.shape}, nulos: {int(_e.isna().sum().sum())}")

In [ ]:
# Backtesting SARIMAX por CP (secuencial). Mismas ordenes + exogenas.
filas = []
for cp in CPS_TODOS:
    serie = get_serie(cp); exog = get_exog(cp)
    s_trv = serie.loc[INI:FIN_VAL];        e_trv = exog.loc[INI:FIN_VAL]
    s_te  = serie.loc[FIN_VAL:].iloc[1:];  e_te = exog.loc[FIN_VAL:].iloc[1:]
    try:
        res = ajustar_sarimax(s_trv, ORDER_SEL, SORDER_SEL, exog=e_trv)
        yt, yp = backtest(res, s_te, STEPS, exog_eval=e_te)
        m = metricas(yt, yp); m['cp'] = cp
        filas.append(m)
        print(f"{cp}: R2={m['r2']:.3f} MAE={m['mae']:.0f} MAPE={m['mape']:.1f}%")
    except Exception as e:
        print(f"{cp}: ERROR {str(e)[:50]}")

sarimax_res = pd.DataFrame(filas)
print()
print('SARIMAX - media 30 CPs limpios:')
print(sarimax_res[['r2', 'mae', 'rmse', 'mape']].mean().round(3))

---
# <font color='#1B3A5C'>  **8. Comparativa de Modelos** </font>

> Tabla resumen con la métrica primaria (R²) y las secundarias, agregadas sobre los 42 CPs.
> Esta misma tabla se ampliará con XGBoost/LightGBM y LSTM/GRU en las fases siguientes.

In [ ]:
resumen = pd.DataFrame({
    'Baseline (naive)': base_res[['r2','mae','rmse','mape']].mean(),
    'SARIMA'          : sarima_res[['r2','mae','rmse','mape']].mean(),
    'SARIMAX'         : sarimax_res[['r2','mae','rmse','mape']].mean(),
}).T.round(3)
resumen.columns = ['R2', 'MAE', 'RMSE', 'MAPE(%)']
print(resumen)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(resumen.index, resumen['R2'], color=[C6, C2, C1], alpha=0.85, edgecolor='white')
for i, v in enumerate(resumen['R2']):
    ax.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')
ax.set_title('R² medio por modelo (test oct-nov 2025, 30 CPs limpios)', fontweight='bold', color=TITULO)
ax.set_ylabel('R²')
plt.tight_layout(); plt.show()

### <font color='#C0392B'><b>8.1 Guardado de resultados</b></font>

In [ ]:
for nombre, tabla in [('baseline', base_res), ('sarima', sarima_res), ('sarimax', sarimax_res)]:
    tabla.to_csv(f'resultados_{nombre}.csv', index=False)
    db[f'resultados_{nombre}'].drop()
    db[f'resultados_{nombre}'].insert_many(tabla.to_dict('records'))
print("Resultados guardados (CSV + MongoDB)")

elapsed = time.time() - start_time
print(f"Tiempo de ejecución: {elapsed/60:.1f} min")

---
# <font color='#1B3A5C'>  **9. Backtesting Multi-Horizonte (24h / 48h / 72h)** </font>

> Requisito de la memoria: horizonte objetivo entre 24 y 72h. Bucle secuencial
> (sin joblib): reusa un fit por modelo y CP y solo varia el STEPS del backtest.
> OJO: re-entrena los modelos, asi que es la celda mas lenta. El 72h coincide con el test.

In [ ]:
# Multi-horizonte 24h/48h/72h (secuencial, sin joblib). Celda lenta: re-entrena.
HORIZONTES = {'24h': 4, '48h': 8, '72h': 12}

filas = []
for cp in CPS_TODOS:
    serie = get_serie(cp); exog = get_exog(cp)
    s_trv = serie.loc[INI:FIN_VAL]; s_te = serie.loc[FIN_VAL:].iloc[1:]
    e_trv = exog.loc[INI:FIN_VAL];  e_te = exog.loc[FIN_VAL:].iloc[1:]
    for nombre, ex_tr, ex_te in [('SARIMA', None, None), ('SARIMAX', e_trv, e_te)]:
        try:
            res = ajustar_sarimax(s_trv, ORDER_SEL, SORDER_SEL, exog=ex_tr)
            for hn, st in HORIZONTES.items():
                yt, yp = backtest(res, s_te, st, exog_eval=ex_te)
                m = metricas(yt, yp); m.update(cp=cp, modelo=nombre, horizonte=hn)
                filas.append(m)
        except Exception as e:
            filas.append({'cp': cp, 'modelo': nombre, 'error': str(e)[:80]})
    print(f'{cp} listo')

multi_res = pd.DataFrame(filas)
multi_res.to_csv('resultados_multihorizonte.csv', index=False)
db['resultados_multihorizonte'].drop()
db['resultados_multihorizonte'].insert_many(multi_res.to_dict('records'))
print(f'Multi-horizonte listo | {len(multi_res)} filas')

In [ ]:
# Lee el CSV generado por la celda anterior (o por run_multihorizonte.py).
multi_res = pd.read_csv('resultados_multihorizonte.csv')
multi_res['cp'] = multi_res['cp'].astype(str).str.zfill(5)
m = multi_res.dropna(subset=['r2'])

med = m.groupby(['modelo', 'horizonte'])['r2'].median().unstack()[['24h', '48h', '72h']].round(3)
avg = m.groupby(['modelo', 'horizonte'])['r2'].mean().unstack()[['24h', '48h', '72h']].round(3)
print('R2 MEDIANA por horizonte (30 CPs limpios):'); print(med)
print(); print('R2 MEDIA por horizonte:'); print(avg)

fig, ax = plt.subplots(figsize=(8, 5))
for modelo, col in [('SARIMA', C2), ('SARIMAX', C1)]:
    ax.plot(['24h', '48h', '72h'], med.loc[modelo].values, marker='o',
            linewidth=2, label=f'{modelo} (mediana)', color=col)
ax.set_xlabel('Horizonte de prediccion'); ax.set_ylabel('R2 mediana (30 CPs limpios)')
ax.set_title('Degradacion del R2 con el horizonte (24h -> 72h)', fontweight='bold', color=TITULO)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
# <font color='#1B3A5C'>  **10. Conclusiones** </font>

> Modelos baseline (a completar tras la ejecución):

- **Naive estacional:** referencia mínima.
- **SARIMA:** captura el ciclo semanal (s=28); limitado en el ciclo diario.
- **SARIMAX:** añade clima + calendario + Fourier diario → debería mejorar a SARIMA.

> Metodología (directiva José):

- Split temporal estricto train/val/test, sin leakage (coherente con el 03).
- Selección de órdenes con control de overfitting (rel_diff ≤ 0.10), no `best_model`.
- Métrica primaria R²; backtesting walk-forward a 72h sobre test oct–nov 2025.

> Pendiente:

- Decidir el tratamiento del CP 08037 (target imputado en val/test).
- Documentar ADF/KPSS y fórmulas de métricas en la memoria (LaTeX/Overleaf).
- Comparar contra XGBoost/LightGBM y LSTM/GRU bajo el **mismo arnés**.